# Nemotron v7.2 — Submission Notebook

Pairs with `nemotron_v72_training.ipynb`.

### Workflow
1. Run the training notebook → download `adapter.zip`
2. Upload `adapter.zip` as a Kaggle dataset (e.g. `nemotron-v72-adapter`)
3. Attach that dataset + this notebook to the competition submission
4. Run this notebook → produces `submission.zip` in `/kaggle/working`

### What this notebook does
- Auto-detects `adapter.zip` (or loose files) from any attached dataset
- Extracts `adapter_config.json` + `adapter_model.safetensors` to `/kaggle/working`
- Verifies the adapter config against eval-server constraints
- Packages the final `submission.zip`


In [ ]:
# ============================================================
# 1. AUTO-DETECT ADAPTER
# ============================================================
import os, json, shutil, zipfile, glob

ADAPTER_ZIP_PATH = None  # set to override auto-detect
WORK_DIR = "/kaggle/working"

if ADAPTER_ZIP_PATH is None:
    candidates = glob.glob("/kaggle/input/*/adapter.zip") + \
                 glob.glob("/kaggle/input/*/*/adapter.zip")
    if candidates:
        ADAPTER_ZIP_PATH = candidates[0]
        print(f"Auto-detected zip: {ADAPTER_ZIP_PATH}")
    else:
        # Fall back to loose adapter files
        loose = glob.glob("/kaggle/input/*/adapter_config.json")
        if loose:
            ADAPTER_ZIP_PATH = os.path.dirname(loose[0])
            print(f"Auto-detected loose adapter dir: {ADAPTER_ZIP_PATH}")
        else:
            raise FileNotFoundError(
                "No adapter.zip or adapter_config.json under /kaggle/input/*. "
                "Attach your adapter dataset to this notebook."
            )

print(f"ADAPTER_ZIP_PATH = {ADAPTER_ZIP_PATH}")


In [ ]:
# ============================================================
# 2. EXTRACT ADAPTER TO /kaggle/working
# ============================================================
REQUIRED = {"adapter_config.json", "adapter_model.safetensors"}

if os.path.isfile(ADAPTER_ZIP_PATH) and ADAPTER_ZIP_PATH.endswith(".zip"):
    with zipfile.ZipFile(ADAPTER_ZIP_PATH) as zf:
        names = zf.namelist()
        print(f"Zip contents ({len(names)} files):")
        for n in names: print(f"  {n}")
        for n in names:
            base = os.path.basename(n)
            if not base: continue
            with zf.open(n) as src, open(os.path.join(WORK_DIR, base), "wb") as dst:
                shutil.copyfileobj(src, dst)
elif os.path.isdir(ADAPTER_ZIP_PATH):
    for fn in os.listdir(ADAPTER_ZIP_PATH):
        fp = os.path.join(ADAPTER_ZIP_PATH, fn)
        if os.path.isfile(fp):
            shutil.copy(fp, os.path.join(WORK_DIR, fn))
            print(f"  copied {fn}")
else:
    raise ValueError(f"Unexpected ADAPTER_ZIP_PATH: {ADAPTER_ZIP_PATH}")

present = set(os.listdir(WORK_DIR)) & REQUIRED
missing = REQUIRED - present
if missing:
    raise RuntimeError(f"Missing required files: {missing}")
print(f"\nAll required files present in {WORK_DIR}")


In [ ]:
# ============================================================
# 3. VERIFY adapter_config.json (eval-server contract)
# ============================================================
cfg_path = os.path.join(WORK_DIR, "adapter_config.json")
with open(cfg_path) as f: cfg = json.load(f)

print(json.dumps(cfg, indent=2))

checks = [
    ("peft_type is LORA",            cfg.get("peft_type") == "LORA"),
    ("r <= 32",                       cfg.get("r", 99) <= 32),
    ("base_model_name canonical",     cfg.get("base_model_name_or_path")
                                        == "metric/nemotron-3-nano-30b-a3b-bf16"),
    ("lora_dropout == 0.0",          cfg.get("lora_dropout", 1.0) == 0.0),
    ("target_modules present",       bool(cfg.get("target_modules"))),
]
print("\nVerification:")
ok = True
for name, passed in checks:
    print(f"  [{'OK' if passed else 'FAIL'}] {name}")
    if not passed: ok = False

if not ok:
    print("\n⚠ Some checks failed — review before submitting.")
else:
    print("\n✓ All checks passed.")


In [ ]:
# ============================================================
# 4. PACKAGE submission.zip
# ============================================================
SUB_ZIP = os.path.join(WORK_DIR, "submission.zip")
if os.path.exists(SUB_ZIP): os.remove(SUB_ZIP)

with zipfile.ZipFile(SUB_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in sorted(REQUIRED):
        fp = os.path.join(WORK_DIR, fn)
        zf.write(fp, arcname=fn)
        print(f"  added {fn}  ({os.path.getsize(fp)/1e6:.2f} MB)")

total_mb = os.path.getsize(SUB_ZIP) / 1024 / 1024
print(f"\nsubmission.zip  ({total_mb:.2f} MB)  -> {SUB_ZIP}")
